DONE
Step 1: Preprocess the data - use the same function across the board
Clearing out the capitalization
Clearing out em dashes, symbols
Clearing out names

TO DO
Step 2: Implementing the features
- Goal: Train a logistic regression on 2 feature representations

Step 3: Implementing the regression based on that data


# Part 1: Imports and Cleaning Text

In [47]:
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline
import numpy as np


In [51]:
# Read in data
train = pd.read_csv('data/train.csv')
val = pd.read_csv('data/val.csv')

In [36]:
# Helper function for cleaning text
def clean_html(text):
    if pd.isna(text):
        return text
    # Remove HTML tags
    clean = re.sub(r'<.*?>', '', str(text))
    # Remove extra whitespaces
    clean = re.sub(r'\s+', ' ', clean).strip()
    # Replace HTML entities
    clean = re.sub(r'&amp;', '&', clean)
    clean = re.sub(r'&lt;', '<', clean)
    clean = re.sub(r'&gt;', '>', clean)
    clean = re.sub(r'&quot;|&#34;', '"', clean)
    clean = re.sub(r'&apos;|&#39;', "'", clean)
    return clean

In [66]:
# use the clean_html function to clean the training data
train['snip'] = train['snip'].apply(clean_html)
val['snip'] = val['snip'].apply(clean_html)

print(train)

                      time                                               snip  \
0      2024-01-01 00:00:00  first of all, it feels like covid again but in...   
1      2024-01-01 00:03:00  to be a software drivenrganization where softw...   
2      2024-01-01 00:15:00  you discuss the power of ai to revolutionize t...   
3      2024-01-01 00:30:00  ai bots like chatgpt and google's bard gained ...   
4      2024-01-01 01:00:00  . >> i could sleep ten hours ai night if i was...   
...                    ...                                                ...   
19868  2024-10-31 21:00:00  cardiovascular science, but they're also pione...   
19869  2024-10-31 21:30:00  i of ai in different fields. have of ai in dif...   
19870  2024-10-31 22:00:00  weighing down on the major averages, both tech...   
19871  2024-10-31 22:23:00  i also think crypto ai that legislation be fro...   
19872  2024-10-31 22:53:00  as we have worked to monitor the adoption iden...   

        channel  word_compl

# Evaulation Metrics

In [60]:
def eval(y_pred, y_true):
    correct = (y_pred == y_true)   # Boolean array: True if correct, False if wrong
    accuracy = correct.sum() / len(y_true)  # Correct / Total
    return accuracy

# Part 2: Implement the 2 features

The goal of these two features is to yield 2 options:
1. Complexity of the vocab - using Tfidvectorizer in sklearn to highlight complex infrequent words and give a complexity score
2. Political leaning - using hugging face models

1. word complexity feature

In [67]:
# Initialize and fit TF-IDF
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(train['snip'])

# Calculate word complexity per snip
word_complexity = X_tfidf.sum(axis=1) / (X_tfidf != 0).sum(axis=1)
word_complexity = np.array(word_complexity).flatten()

# Add it to the train dataframe
train['word_complexity'] = word_complexity

print(train)

                      time                                               snip  \
0      2024-01-01 00:00:00  first of all, it feels like covid again but in...   
1      2024-01-01 00:03:00  to be a software drivenrganization where softw...   
2      2024-01-01 00:15:00  you discuss the power of ai to revolutionize t...   
3      2024-01-01 00:30:00  ai bots like chatgpt and google's bard gained ...   
4      2024-01-01 01:00:00  . >> i could sleep ten hours ai night if i was...   
...                    ...                                                ...   
19868  2024-10-31 21:00:00  cardiovascular science, but they're also pione...   
19869  2024-10-31 21:30:00  i of ai in different fields. have of ai in dif...   
19870  2024-10-31 22:00:00  weighing down on the major averages, both tech...   
19871  2024-10-31 22:23:00  i also think crypto ai that legislation be fro...   
19872  2024-10-31 22:53:00  as we have worked to monitor the adoption iden...   

        channel  word_compl

In [68]:
# Transform validation snips using the same TF-IDF vectorizer
X_val_tfidf = vectorizer.transform(val['snip'])

# Compute complexity
val_word_complexity = X_val_tfidf.sum(axis=1) / (X_val_tfidf != 0).sum(axis=1)
val_word_complexity = np.array(val_word_complexity).flatten()

# Add to val DataFrame
val['word_complexity'] = val_word_complexity

print(val)

                     time                                               snip  \
0     2024-11-01 01:00:00  . ♪ >> there's a kyu cho right have things tha...   
1     2024-11-01 02:42:00  he says the ai tool helped create a new fronti...   
2     2024-11-01 03:00:00  . >> the all new godaddy arrow put your busine...   
3     2024-11-01 03:26:00  in some cases they are powered by generative a...   
4     2024-11-01 03:30:00  this was a ivotal it comes to ai. this was a p...   
...                   ...                                                ...   
3034  2024-12-31 15:00:00  however, the ai trade is only one part of the ...   
3035  2024-12-31 16:13:00  oz but also was highlighted as a product by cr...   
3036  2024-12-31 17:00:00  the all new godaddy airo helps you get your bu...   
3037  2024-12-31 18:00:00  we are going to be way ahead on ai. we have to...   
3038  2024-12-31 20:00:00  his fourth management role after spells at der...   

        channel  word_complexity  
0   

In [69]:
# X = features, y = labels
x_train = train[['word_complexity']]  # Needs to be 2D
y_train = train['channel']

x_val = val[['word_complexity']]
y_val = val['channel']

In [70]:
from sklearn.linear_model import LogisticRegression

# Train
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

# Predict
val_preds = model.predict(x_val)

# Evaluate
val_accuracy = eval(val_preds, y_val)
print(val_accuracy)

0.07535373478117802


feature 2 hugging face
can't be used because of the lack of access to the repository

In [ ]:
clf = pipeline(
    "text-classification",
    model="allenai/political-leaning",
    return_all_scores=True,  # get scores for left/center/right
    truncation=True,
    max_length=512
)

# 3) Convert each snip into a dict of {label: score}
def get_lean_scores(text):
    out = clf(text)[0]
    return {item['label']: item['score'] for item in out}

# build DataFrames of lean scores
lean_tr = pd.DataFrame(list(map(get_lean_scores, train['snip'])))
lean_val = pd.DataFrame(list(map(get_lean_scores, val['snip'])))